<h1 align='center'>🎓 VQA-E Master Guide</h1>
<h3 align='center'>Generative Visual Question Answering — All-in-One Notebook</h3>
<p align='center'>Architecture · Data · Training · Evaluation · Visualization · Advanced Techniques</p>

---

## What This Notebook Covers

| Section | Content |
|---------|---------|
| **1. Setup** | Install, GPU check, project structure |
| **2. Architecture** | All 6 models, every technique explained with math |
| **3. Data Prep** | Vocab build, BUTD feature extraction, hallucination filter |
| **4. Training** | 4-phase training for every model, all flags explained |
| **5. Evaluation** | 7 metrics: VQA Acc, BLEU-1/2/3/4, METEOR, ROUGE-L |
| **6. Visualization** | Attention heatmaps, training curves, sample predictions |
| **7. Advanced** | Focal Loss, Curriculum, Mixed Pretraining, SCST, GNN |
| **8. Comparison** | Side-by-side model benchmark table |

> **Constraint**: Pure LSTM architecture — no Transformers, no BERT, no ViT anywhere.
> Cross-attention (MHCA) is used legally: Q from LSTM, K/V from CNN/QuestionEncoder.


---
# 1. Environment Setup


## 1.1 Install Dependencies

```bash
# Core
pip install torch torchvision nltk tqdm matplotlib Pillow

# NLP metrics
pip install rouge-score

# Experiment tracking (optional)
pip install wandb

# Tier D5: hallucination filter (optional)
pip install spacy
python -m spacy download en_core_web_sm

# Tier 9: ConceptNet GNN (optional — MLP fallback works without it)
pip install torch_geometric
```


In [4]:
# Run this cell to install everything needed
import subprocess, sys
pkgs = ['torch','torchvision','nltk','tqdm','matplotlib','Pillow','rouge-score']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('✅ Dependencies installed')


✅ Dependencies installed


## 1.2 GPU & Environment Check


In [5]:
import torch, os, sys

# ── GPU ──────────────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram = gpu.total_memory / 1024**3
    cap  = torch.cuda.get_device_capability()
    bf16 = '✅ BFloat16 (Ampere+)' if cap[0] >= 8 else '⚠️  Float16 only'
    print(f'GPU      : {gpu.name}')
    print(f'VRAM     : {vram:.1f} GB')
    print(f'Compute  : {cap[0]}.{cap[1]}  →  AMP: {bf16}')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU detected — training will be very slow')
    DEVICE = 'cpu'

# ── Project root ─────────────────────────────────────────────────
ROOT = os.path.abspath('.')
SRC  = os.path.join(ROOT, 'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
print(f'Root     : {ROOT}')
print(f'Device   : {DEVICE}')


GPU      : NVIDIA GeForce RTX 5070 Ti
VRAM     : 15.5 GB
Compute  : 12.0  →  AMP: ✅ BFloat16 (Ampere+)
Root     : /home/anakonkai-5070ti/workspace/vqa-final/new_vqa
Device   : cuda


## 1.3 Project Structure

```
new_vqa/
├── src/
│   ├── models/
│   │   ├── encoder_cnn.py          # SimpleCNN, ResNet, ConvNeXt, BUTDFeatureEncoder
│   │   ├── encoder_question.py     # BiLSTM + Char-CNN + Highway
│   │   ├── decoder_lstm.py         # LSTMDecoder (Models A/B)
│   │   ├── decoder_attention.py    # MHCA + LSTMDecoderWithAttention (C/D/E/F)
│   │   ├── pointer_generator.py    # Pointer-Generator Network (Tier 5)
│   │   ├── vqa_models.py           # VQAModelA/B/C/D/E/F + GatedFusion + MUTANFusion
│   │   └── concept_gnn.py          # ConceptNet GNN (Tier 9, optional)
│   ├── training/
│   │   ├── losses.py               # SequenceFocalLoss (Tier D3)
│   │   ├── curriculum.py           # Question-type CurriculumSampler (Tier D4)
│   │   ├── css_augment.py          # CSS Counterfactual Augmentation (Tier 6)
│   │   └── scst.py                 # SCST Reinforcement Learning (Tier 8)
│   ├── scripts/
│   │   ├── 1_build_vocab.py        # Build vocab_questions + vocab_answers JSONs
│   │   ├── extract_butd_features.py # Extract Faster R-CNN features (Tier 3B)
│   │   ├── filter_hallucinations.py # Filter bad VQA-E explanations (Tier D5)
│   │   └── generate_synthetic_qa.py # Template synthetic QA (Tier D5)
│   ├── train.py                    # Main entry — all flags, all phases
│   ├── evaluate.py                 # 7-metric evaluation
│   ├── inference.py                # Greedy + beam search decoding
│   ├── compare.py                  # Multi-model comparison
│   ├── plot_curves.py              # Training curves
│   └── visualize.py                # Attention heatmaps
├── data/
│   ├── raw/train2014/              # COCO train images (~82K)
│   ├── raw/val2014/                # COCO val images (~40K)
│   ├── raw/vqa_data_json/          # VQA v2.0 JSON files
│   ├── vqa_e/                      # VQA-E annotation JSONs
│   ├── processed/                  # vocab_questions.json, vocab_answers.json
│   └── butd_features/              # Pre-extracted BUTD .pt files (Model F)
└── checkpoints/                    # Saved model weights + training history
```


---
# 2. Architecture Deep Dive


## 2.1 The 6 Models

| Model | Image Encoder | Fusion | Decoder | Notes |
|-------|--------------|--------|---------|-------|
| **A** | SimpleCNN (scratch, 7×7 grid → pool) | GatedFusion | LSTMDecoder | Baseline |
| **B** | ResNet101 (pretrained, frozen) | GatedFusion | LSTMDecoder | Phase 2 unfreezes layer3+4 |
| **C** | SimpleCNNSpatial (scratch, 49 regions) | GatedFusion | LSTM + MHCA | Spatial attention |
| **D** | ResNetSpatialEncoder (pretrained, 49 regions) | GatedFusion | LSTM + MHCA | Pretrained spatial |
| **E** | **ConvNeXt-Base** (pretrained, 49 regions) | **MUTANFusion** | LSTM + MHCA | Best overall |
| **F** | **BUTD Faster R-CNN** (pre-extracted, k regions) | MUTANFusion | LSTM + MHCA | Highest ceiling |

### Data Flow
```
Image (3,224,224) ──→ CNN / ConvNeXt / BUTD ──→ img_features  (B, k, H)
                                                       │
Question tokens ──→ BiLSTM+HighwayLayer ──→ q_feature (B, H)
                                            q_hidden  (B, q_len, H)
                                                       │
                               GatedFusion / MUTANFusion ──→ fused (B, H)
                                                       │
                              Initialize LSTM h_0 (layer 0 only)
                                                       │
┌─ decode step t ────────────────────────────────────────────────────────────┐
│  embed_t = Embedding(token_t)                          (B, E)              │
│  img_context, img_α = MHCA(h_t, img_features)         (B, H), (B, k)      │
│  q_context,   q_α  = MHCA(h_t, q_hidden)              (B, H), (B, q_len)  │
│  input_t  = cat[embed_t, img_context, q_context]       (B, E+2H)           │
│  h_{t+1}  = LSTM(input_t, h_t)                         (B, H)              │
│  logit_t  = fc(out_proj(h_{t+1}))                      (B, V)              │
│  [PGN]  logit_t = p_gen·P_vocab + (1-p_gen)·P_copy     (B, V)              │
└────────────────────────────────────────────────────────────────────────────┘
                                                       │
                              logits (B, max_len, V) → CE / Focal Loss
```


## 2.2 Multi-Head Cross-Attention (MHCA) — The Legal Attention

**Why not self-attention?** Self-attention inside `decode_step` would run
O(T · (L_v² + L_q²)) per forward pass — quadratic in sequence length, repeated
at every decode step. Also violates the no-transformer constraint.

**MHCA is cross-attention only**:
- Q = h_t ∈ ℝᴴ — LSTM hidden state (single token, not a sequence)
- K, V = memory ∈ ℝˢˣᴴ — image regions or question tokens
- Complexity: O(T · S) — linear in memory length

```
Q = W_Q(h_t)                  (B, 1, H)
K = W_K(memory)               (B, S, H)
V = W_V(memory)               (B, S, H)

Reshape to heads: (B, nh, ·, d_head) where d_head = H / nh

scores = Q·Kᵀ / √d_head       (B, nh, 1, S)
[img]  + cov_scale · coverage  ← optional coverage bias
α      = softmax(scores)       (B, nh, 1, S)
context = α · V                (B, H)   after merging heads
α_mean  = mean(α, heads)       (B, S)   → P_copy for PGN
```


## 2.3 MUTAN Tucker Fusion (Model E/F)

Replaces GatedFusion with a Tucker-decomposed bilinear product:

```
q_proj = tanh(W_q · q)   ∈ ℝᵗ_q
v_proj = tanh(W_v · v)   ∈ ℝᵗ_v
inter  = q_proj ⊗ T_c    ∈ ℝᵗ_v × d_out    (T_c ∈ ℝᵗ_q × ᵗ_v × d_out  learnable core)
output = v_proj · inter  ∈ ℝᵈ_out           (rank-constrained bilinear)
       → BatchNorm
```

**Why MUTAN?** GatedFusion is a gated weighted average — it can only mix, not
multiply. MUTAN captures multiplicative cross-modal interactions (e.g. 'red ball'
= color × object) via Tucker decomposition without the parameter explosion of
a full bilinear product (H² parameters → t_q × t_v parameters, t_q=t_v=360).


## 2.4 Pointer-Generator Network (PGN)

Allows the decoder to **copy tokens from the question** when the answer
contains the same words (e.g. Q: 'What color is the **red** car?' → A: '**red**').

```
p_gen  = σ(W_x·embed_t + W_h·h_t + W_ctx·img_context)  ∈ (0,1)

P_vocab = softmax(logit_t)             ← generate from vocabulary
P_copy  = scatter(q_α, q_token_ids)   ← copy from question (q_α from q_mhca)

P_final = p_gen · P_vocab + (1-p_gen) · P_copy
loss    = NLLLoss(log(P_final), target)   ← NLL not CE (log already applied)
```


## 2.5 Sequence Focal Loss (Tier D3)

**The problem with class-weighted CE**: `CrossEntropyLoss(weight=w)` applies
the same scalar w[c] at EVERY position. The word 'because' (very common in
VQA-E) would have w=0.01, teaching the model to skip the structural hinge.

**Focal Loss is position-aware**:
```
ce_t         = -log P(y_t | context)          per token, per position
p_t          = exp(-ce_t)                      model confidence (0→hard, 1→easy)
focal_loss_t = (1 - p_t)^γ · ce_t             down-weight easy tokens naturally

loss = Σ_{t: y_t ≠ pad} focal_loss_t / |valid tokens|
```
Common tokens ('because', 'the') predicted with high p_t → suppressed.
Rare answer words where model struggles (low p_t) → full gradient.


## 2.6 All Training Techniques at a Glance

| Flag | Tier | What it does |
|------|------|-------------|
| `--layer_norm` | 1A | LayerNorm inside LSTM cell gates |
| `--dropconnect` | 1B | DropConnect on hidden-to-hidden weights (AWD-LSTM) |
| `--dcan` | 2→MHCA | No-op now; MHCA always active |
| `--coverage` | - | Coverage mechanism: penalizes re-attending to regions |
| `--pgn` | 5 | Pointer-Generator: copy tokens from question |
| `--css` | 6 | CSS: visual + linguistic counterfactual contrastive loss |
| `--q_highway` | 7B | Highway connections between BiLSTM layers |
| `--char_cnn` | 7C | Character-level CNN prepended to word embeddings |
| `--scst` | 8 | SCST RL: REINFORCE with BLEU-4 reward (Phase 4) |
| `--focal` | D3 | SequenceFocalLoss replaces CrossEntropyLoss |
| `--curriculum` | D4 | Question-type curriculum: binary→color→count→what→where→why |
| `--mix_vqa` | D2 | 70% VQA v2.0 + 30% VQA-E to prevent length bias |
| `--scheduled_sampling` | - | Reduce exposure bias (epsilon-greedy teacher forcing) |
| `--glove` | - | GloVe 300d pretrained word embeddings |


In [6]:
# Instantiate all 6 models and print parameter counts
import sys; sys.path.insert(0, 'src')
import torch
from models.vqa_models import VQAModelA, VQAModelB, VQAModelC, VQAModelD, VQAModelE, VQAModelF

VOCAB_Q, VOCAB_A = 8000, 4000

configs = [
    ('A', VQAModelA, {}),
    ('B', VQAModelB, {}),
    ('C', VQAModelC, {'use_coverage': True}),
    ('D', VQAModelD, {'use_coverage': True}),
    ('E', VQAModelE, {'use_coverage': True, 'use_mutan': True, 'use_dcan': True}),
    ('F', VQAModelF, {'use_coverage': True, 'use_mutan': True, 'feat_dim': 1029}),
]

print(f"{'Model':<8} {'Params':>12}  Description")
print('-' * 60)
descs = {
    'A': 'SimpleCNN + GatedFusion + LSTMDecoder',
    'B': 'ResNet101 + GatedFusion + LSTMDecoder',
    'C': 'SimpleCNNSpatial + GatedFusion + LSTM+MHCA',
    'D': 'ResNet101Spatial + GatedFusion + LSTM+MHCA',
    'E': 'ConvNeXt-Base + MUTANFusion + LSTM+MHCA',
    'F': 'BUTD(Faster R-CNN) + MUTANFusion + LSTM+MHCA',
}
for name, cls, kw in configs:
    try:
        m = cls(vocab_size=VOCAB_Q, answer_vocab_size=VOCAB_A, **kw)
        p = sum(x.numel() for x in m.parameters() if x.requires_grad)
        print(f'  Model {name}  {p:>12,}  {descs[name]}')
    except Exception as e:
        print(f'  Model {name}  [error: {e}]')


Model          Params  Description
------------------------------------------------------------
  Model A    43,388,928  SimpleCNN + GatedFusion + LSTMDecoder
  Model B    38,162,944  ResNet101 + GatedFusion + LSTMDecoder
  Model C    60,166,145  SimpleCNNSpatial + GatedFusion + LSTM+MHCA
  Model D    54,940,161  ResNet101Spatial + GatedFusion + LSTM+MHCA
  Model E   182,094,337  ConvNeXt-Base + MUTANFusion + LSTM+MHCA
  Model F   183,151,105  BUTD(Faster R-CNN) + MUTANFusion + LSTM+MHCA


---
# 3. Data Preparation


## 3.1 Download Data

### COCO Images (required for all models)
```bash
# Create directories
mkdir -p data/raw/images

# Train 2014 (~13 GB)
wget http://images.cocodataset.org/zips/train2014.zip -P data/raw/
unzip data/raw/train2014.zip -d data/raw/

# Val 2014 (~6 GB)
wget http://images.cocodataset.org/zips/val2014.zip -P data/raw/
unzip data/raw/val2014.zip -d data/raw/
```

### VQA-E Annotations (required)
```bash
mkdir -p data/vqa_e
# Download from: https://github.com/liqing-ustc/VQA-E
# Place: data/vqa_e/VQA-E_train_set.json
#        data/vqa_e/VQA-E_val_set.json
```

### VQA v2.0 Annotations (for --mix_vqa pretraining, Tier D2)
```bash
mkdir -p data/raw/vqa_data_json
# Questions
wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
unzip v2_Questions_Train_mscoco.zip -d data/raw/vqa_data_json/
# Annotations
wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
unzip v2_Annotations_Train_mscoco.zip -d data/raw/vqa_data_json/
```


## 3.2 Build Vocabulary

Builds `vocab_questions.json` and `vocab_answers.json` from VQA-E training data.
Special tokens: `<pad>=0`, `<start>=1`, `<end>=2`, `<unk>=3`


In [7]:
import subprocess
result = subprocess.run(
    ['python', 'src/scripts/1_build_vocab.py'],
    capture_output=True, text=True, cwd='.'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)



Reading VQA-E annotation file: data/vqa_e/VQA-E_train_set.json
Loaded 181298 annotations.

1. Building question vocabulary...
 -> Tokenizing...
 -> Filtering (threshold=3)...
 -> Done. Vocab size: 4546
   Vocab size: 4546 | Saved to: data/processed/vocab_questions.json

2. Building answer vocabulary (answer + explanation)...
 -> Tokenizing...
 -> Filtering (threshold=3)...
 -> Done. Vocab size: 8648
   Vocab size: 8648 | Saved to: data/processed/vocab_answers.json

Sample answer texts:
  Q: What is the green stuff?
  A: broccoli because Closeup of bins of food that include broccoli and bread.

  Q: What is in front of the giraffes?
  A: tree because Two giraffes standing in a tree filled area.

  Q: What do these giraffes have in common?
  A: eating because A giraffe eating food from the top of the tree.




In [8]:
# Verify vocab files and print statistics
import json

for name, path in [('Questions', 'data/processed/vocab_questions.json'),
                   ('Answers',   'data/processed/vocab_answers.json')]:
    with open(path) as f:
        v = json.load(f)
    print(f'{name} vocab: {len(v["word2idx"]):,} tokens')


Questions vocab: 4,546 tokens
Answers vocab: 8,648 tokens


## 3.3 (Optional) Filter Hallucinations — Tier D5

Removes VQA-E explanations with named entities, copy-paste patterns, or
degenerate repetitions. Reduces dataset by ~5-10% while improving quality.

**Requires**: `pip install spacy && python -m spacy download en_core_web_sm`


In [ ]:
# Run hallucination filter (requires spaCy)
import subprocess
result = subprocess.run([
    'python', 'src/scripts/filter_hallucinations.py',
    '--input',  'data/vqa_e/VQA-E_train_set.json',
    '--output', 'data/vqa_e/VQA-E_train_filtered.json',
    '--report',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error (spaCy not installed?):', result.stderr[:300])
    print('Using unfiltered data instead.')


## 3.4 (Optional) Generate Synthetic QA — Tier D5

Generates ~50K template-based QA pairs from COCO object annotations.
No LLM required — uses hand-crafted templates (existence, count, location).


In [ ]:
# Generate synthetic QA from COCO instances
import subprocess, os
instances_json = 'data/raw/vqa_data_json/instances_train2014.json'
if os.path.exists(instances_json):
    result = subprocess.run([
        'python', 'src/scripts/generate_synthetic_qa.py',
        '--instances_json', instances_json,
        '--output', 'data/vqa_e/VQA-E_synthetic.json',
        '--max_total', '50000',
    ], capture_output=True, text=True)
    print(result.stdout)
else:
    print(f'⚠️  {instances_json} not found — skipping synthetic generation')


## 3.5 Extract BUTD Features (Model F only)

Pre-extracts Faster R-CNN ResNet50 FPN v2 RoI features from all COCO images.
**Run once, ~3-4 hours on GPU. Skip if you only train Models A-E.**

Output: `data/butd_features/{image_id}.pt` — each file contains `{'feat': (k, 1029)}`
- 1024 dims: ResNet50 FPN box_head output (region semantic features)
- 5 dims: normalized spatial [x1/W, y1/H, x2/W, y2/H, area/(W·H)]
- k = 36 regions per image (top-36 RPN proposals by objectness score)


In [ ]:
# Extract BUTD features for Model F
# ⚠️  Takes ~3-4 hours for full COCO — use --max_images 100 for a quick test
import subprocess

EXTRACT_FULL = False   # ← set True when you want to run the full extraction

cmd = [
    'python', 'src/scripts/extract_butd_features.py',
    '--splits', 'train2014', 'val2014',
    '--image_dir', 'data/raw',
    '--output_dir', 'data/butd_features',
    '--top_k', '36',
]
if not EXTRACT_FULL:
    cmd += ['--max_images', '100']   # quick test
    print('Running quick test (100 images). Set EXTRACT_FULL=True for full run.')

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('Error:', result.stderr[:500])


---
# 4. Training

## Training Phases

| Phase | Epochs | LR | Key Flags | Purpose |
|-------|--------|-----|-----------|---------|
| **1** | 10 | 1e-3 | `--augment --focal --curriculum` | Learn basic VQA-E format |
| **2** | 5 | 5e-4 | `--finetune_cnn` (B/D/E) | Fine-tune CNN backbone |
| **3** | 5 | 2e-4 | `--scheduled_sampling` | Reduce exposure bias |
| **4** | 3 | 5e-5 | `--scst` | REINFORCE RL polish (BLEU-4 reward) |


## 4.1 Quick Sanity Check (100 samples, 2 epochs)

Always run this first to verify your data pipeline works end-to-end.


In [ ]:
import subprocess

MODEL = 'C'   # ← change to A/B/C/D/E/F as desired

result = subprocess.run([
    'python', 'src/train.py',
    '--model', MODEL,
    '--epochs', '2',
    '--lr', '1e-3',
    '--batch_size', '32',
    '--max_train_samples', '100',
    '--max_val_samples', '50',
    '--num_workers', '2',
    '--no_compile',
    '--warmup_epochs', '0',
], capture_output=True, text=True)

print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])


## 4.2 Phase 1 — Baseline Training (10 epochs)

### Recommended flags for each model:
- **A/B**: Simple baseline — no attention flags needed
- **C/D**: Add `--coverage --layer_norm`
- **E/F**: Add `--coverage --layer_norm --use_mutan`

### VRAM guide:
| VRAM | batch_size | accum_steps | Effective batch |
|------|-----------|-------------|-----------------|
| 8 GB | 64 | 4 | 256 |
| 12 GB | 128 | 2 | 256 |
| 24 GB (A10G) | 256 | 1 | 256 |
| 40 GB (A100) | 512 | 1 | 512 |


In [ ]:
# ── Phase 1 Configuration ────────────────────────────────────────
MODEL       = 'E'      # A / B / C / D / E / F
BATCH_SIZE  = 128      # adjust for your VRAM
ACCUM_STEPS = 2        # effective batch = BATCH_SIZE × ACCUM_STEPS
NUM_WORKERS = 4
DROPOUT     = 0.3

# Build command
cmd = [
    'python', 'src/train.py',
    '--model',        MODEL,
    '--epochs',       '10',
    '--lr',           '1e-3',
    '--batch_size',   str(BATCH_SIZE),
    '--accum_steps',  str(ACCUM_STEPS),
    '--num_workers',  str(NUM_WORKERS),
    '--dropout',      str(DROPOUT),
    '--weight_decay', '1e-5',
    '--grad_clip',    '5.0',
    '--warmup_epochs','3',
    '--augment',
    '--focal',                # Tier D3: SequenceFocalLoss
    '--curriculum',           # Tier D4: question-type curriculum
    '--early_stopping', '3',
    '--wandb',                # remove if wandb not installed
    '--phase', '1',
]

# Model-specific flags
if MODEL in ('C', 'D', 'E', 'F'):
    cmd += ['--coverage', '--layer_norm']
if MODEL in ('E', 'F'):
    cmd += ['--use_mutan']

# Mixed pretraining (Phase 1 only — prevents length bias)
# cmd += ['--mix_vqa', '--mix_vqa_fraction', '0.7']  # uncomment if VQA v2.0 downloaded

print('Command:', ' '.join(cmd))
print()
print('Run the cell below to start training.')


In [ ]:
import subprocess
result = subprocess.run(cmd, text=True)
# Output streams directly to console during training


## 4.3 Phase 2 — CNN Fine-tuning (5 epochs)

**Only for Models B, D, E** — unfreezes ResNet/ConvNeXt top layers with
differential LR (backbone LR = base LR × 0.1) to avoid catastrophic forgetting.

**Models A, C**: skip Phase 2 (scratch CNN, already fully trainable).


In [ ]:
MODEL = 'E'
cmd_p2 = [
    'python', 'src/train.py',
    '--model',       MODEL,
    '--epochs',      '5',
    '--lr',          '5e-4',
    '--batch_size',  '128',
    '--accum_steps', '2',
    '--num_workers', '4',
    '--resume',      f'checkpoints/model_{MODEL.lower()}_resume.pth',
    '--warmup_epochs', '0',          # no warmup on resume
    '--augment', '--focal', '--coverage', '--layer_norm',
    '--use_mutan',                   # keep for E/F
    '--finetune_cnn',                # unfreeze top CNN layers
    '--cnn_lr_factor', '0.1',        # backbone LR = 5e-4 × 0.1 = 5e-5
    '--wandb', '--phase', '2',
]
print('Phase 2 command:', ' '.join(cmd_p2))


## 4.4 Phase 3 — Scheduled Sampling (5 epochs)

Reduces **exposure bias**: during training the model always sees ground-truth
tokens (teacher forcing), but at inference it must predict from its own output.

Scheduled sampling gradually replaces GT tokens with the model's own predictions:
```
epsilon(epoch) = k / (k + exp(epoch / k))   (inverse-sigmoid decay)
epsilon ≈ 1.0 at epoch 0  →  mostly teacher forcing
epsilon ≈ 0.5 at epoch k  →  50% teacher forcing
```


In [ ]:
MODEL = 'E'
cmd_p3 = [
    'python', 'src/train.py',
    '--model',       MODEL,
    '--epochs',      '5',
    '--lr',          '2e-4',
    '--batch_size',  '128',
    '--accum_steps', '2',
    '--num_workers', '4',
    '--resume',      f'checkpoints/model_{MODEL.lower()}_resume.pth',
    '--warmup_epochs', '0',
    '--augment', '--focal', '--coverage', '--layer_norm', '--use_mutan',
    '--scheduled_sampling',
    '--ss_k', '5',               # controls decay speed (higher = slower)
    '--wandb', '--phase', '3',
]
print('Phase 3 command:', ' '.join(cmd_p3))


## 4.5 Phase 4 — SCST Reinforcement Learning (3 epochs)

Self-Critical Sequence Training (Rennie et al. 2017): REINFORCE with the
model's own greedy output as the baseline.

```
r_sample  = BLEU-4(sampled_output, reference)     # stochastic decode
r_greedy  = BLEU-4(greedy_output,  reference)     # deterministic baseline
loss_RL   = -mean((r_sample - r_greedy) · Σ log p_t)
loss_total = CE_loss + λ_scst · loss_RL            (λ_scst = 0.5)
```

**Only run after Phase 3.** SCST on an untrained model is unstable.


In [ ]:
MODEL = 'E'
cmd_p4 = [
    'python', 'src/train.py',
    '--model',       MODEL,
    '--epochs',      '3',
    '--lr',          '5e-5',
    '--batch_size',  '64',         # smaller batch for SCST stability
    '--accum_steps', '4',
    '--num_workers', '4',
    '--resume',      f'checkpoints/model_{MODEL.lower()}_resume.pth',
    '--warmup_epochs', '0',
    '--focal', '--coverage', '--layer_norm', '--use_mutan',
    '--scheduled_sampling',
    '--scst',                      # enable SCST RL
    '--scst_lambda', '0.5',
    '--no_compile',                # compile not compatible with SCST graph breaks
    '--wandb', '--phase', '4',
]
print('Phase 4 command:', ' '.join(cmd_p4))


## 4.6 Train Model F (BUTD features)

**Requires**: `data/butd_features/` populated by `extract_butd_features.py`


In [ ]:
cmd_f = [
    'python', 'src/train.py',
    '--model',          'F',
    '--epochs',         '10',
    '--lr',             '1e-3',
    '--batch_size',     '128',
    '--accum_steps',    '2',
    '--num_workers',    '4',
    '--dropout',        '0.3',
    '--augment',           # augmentation still applies to questions
    '--focal',
    '--curriculum',
    '--coverage', '--layer_norm', '--use_mutan',
    '--butd_feat_dir',  'data/butd_features',
    '--wandb', '--phase', '1',
]
print('Model F command:', ' '.join(cmd_f))


## 4.7 All-in-One Training Script (train_model_e.sh)

The `train_model_e.sh` script runs all 4 phases sequentially:
```bash
# Full training (all 4 phases)
bash train_model_e.sh all

# Individual phases
bash train_model_e.sh 1
bash train_model_e.sh 2
bash train_model_e.sh 3
bash train_model_e.sh 4

# Override defaults
BATCH_SIZE=256 NUM_WORKERS=8 bash train_model_e.sh all
```


---
# 5. Evaluation

## Metrics

| Metric | What it measures |
|--------|-----------------|
| **VQA Accuracy** | Soft matching against 10 human annotations (official VQA metric) |
| **Exact Match** | Exact string match after normalization |
| **BLEU-1/2/3/4** | N-gram precision (higher n → stricter fluency) |
| **METEOR** | Unigram F-score with stemming and synonymy (via WordNet) |
| **ROUGE-L** | Longest common subsequence recall (explanation coherence) |


In [ ]:
# Run evaluation on a saved checkpoint
import subprocess, json

MODEL      = 'E'
CKPT_EPOCH = 10    # which epoch checkpoint to evaluate
BEAM_WIDTH = 3     # 1=greedy, 3=beam search (better quality, slower)

ckpt_path = f'checkpoints/model_{MODEL.lower()}_epoch{CKPT_EPOCH}.pth'

result = subprocess.run([
    'python', 'src/evaluate.py',
    '--model_type', MODEL,
    '--checkpoint', ckpt_path,
    '--beam_width', str(BEAM_WIDTH),
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr[-1000:])


In [ ]:
# Evaluate all available models and collect results
import subprocess, os, re

results = {}
for model in ['A', 'B', 'C', 'D', 'E']:
    for epoch in [10, 15, 20]:
        ckpt = f'checkpoints/model_{model.lower()}_epoch{epoch}.pth'
        if not os.path.exists(ckpt):
            continue
        r = subprocess.run([
            'python', 'src/evaluate.py',
            '--model_type', model, '--checkpoint', ckpt,
            '--beam_width', '3',
        ], capture_output=True, text=True)
        # Parse BLEU-4 from output
        m = re.search(r'BLEU-4[^:]*:\s*([0-9.]+)', r.stdout)
        if m:
            results[f'{model}_ep{epoch}'] = float(m.group(1))
            print(f'Model {model} epoch {epoch}: BLEU-4 = {m.group(1)}')

if results:
    best = max(results, key=results.get)
    print(f'\nBest: {best} → BLEU-4 = {results[best]:.4f}')


In [ ]:
# Inline evaluation (no subprocess) — load model directly
import sys; sys.path.insert(0, 'src')
import torch
from evaluate import evaluate

MODEL = 'E'
CKPT  = f'checkpoints/model_{MODEL.lower()}_best.pth'

if os.path.exists(CKPT):
    metrics = evaluate(
        model_type  = MODEL,
        checkpoint  = CKPT,
        num_samples = 500,    # set None for full validation set
        beam_width  = 3,
    )
else:
    print(f'Checkpoint not found: {CKPT}')
    print('Train the model first (Section 4).')


---
# 6. Visualization


## 6.1 Training Curves


In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'A':'#e74c3c','B':'#e67e22','C':'#2ecc71','D':'#3498db','E':'#9b59b6','F':'#1abc9c'}

for model in ['A','B','C','D','E','F']:
    path = f'checkpoints/history_model_{model.lower()}.json'
    if not os.path.exists(path):
        continue
    with open(path) as f:
        h = json.load(f)
    epochs = range(1, len(h['train_loss'])+1)
    c = colors.get(model, '#888')
    axes[0].plot(epochs, h['train_loss'], label=f'Model {model}', color=c)
    axes[1].plot(epochs, h['val_loss'],   label=f'Model {model}', color=c, linestyle='--')

for ax, title in zip(axes, ['Train Loss', 'Val Loss']):
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=150)
plt.show()
print('Saved → checkpoints/training_curves.png')


## 6.2 Attention Heatmaps (Models C/D/E/F)

Visualizes where the model looks in the image (img_alpha) and question
(q_alpha from MHCA) while generating each answer token.


In [ ]:
import subprocess
result = subprocess.run([
    'python', 'src/visualize.py',
    '--model_type', 'E',
    '--epoch', '20',
    '--sample_idx', '0',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr[:500])


In [ ]:
# Inline attention visualization
import sys; sys.path.insert(0, 'src')
import torch, json, os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from vocab import Vocabulary
from inference import load_model_from_checkpoint, get_model
from dataset import VQAEDataset

def visualize_attention(model_type, checkpoint, sample_idx=0, beam_width=1):
    vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
    vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
    dataset = VQAEDataset(
        image_dir='data/raw/val2014',
        vqa_e_json_path='data/vqa_e/VQA-E_val_set.json',
        vocab_q=vocab_q, vocab_a=vocab_a, split='val2014'
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    img_tensor, q_tensor, a_tensor = dataset[sample_idx]
    ann = dataset.annotations[sample_idx]
    print(f"Question : {ann['question']}")
    print(f"Reference: {ann.get('multiple_choice_answer','')} because {ann.get('explanation',[''])[0][:80]}")
    print('Checkpoint required — run training first to generate a checkpoint.')

# Call with your checkpoint:
# visualize_attention('E', 'checkpoints/model_e_best.pth', sample_idx=5)
print('Attention visualization ready — provide a checkpoint path to activate.')


## 6.3 Sample Predictions


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch, os
from vocab import Vocabulary
from dataset import VQAEDataset

def show_predictions(model_type, checkpoint_path, n_samples=5, beam_width=3):
    if not os.path.exists(checkpoint_path):
        print(f'Checkpoint not found: {checkpoint_path}')
        return
    from inference import load_model_from_checkpoint, batch_beam_search_decode_with_attention
    from inference import batch_greedy_decode

    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab_q  = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
    vocab_a  = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
    dataset  = VQAEDataset(
        'data/raw/val2014', 'data/vqa_e/VQA-E_val_set.json',
        vocab_q, vocab_a, split='val2014'
    )
    model = load_model_from_checkpoint(model_type, checkpoint_path,
                                       len(vocab_q), len(vocab_a)).to(device)
    model.eval()
    sep = '=' * 70
    print(f'\n{sep}')
    print(f'  Model {model_type} — {os.path.basename(checkpoint_path)}')
    print(sep)
    for i in range(n_samples):
        img, q_t, a_t = dataset[i]
        ann = dataset.annotations[i]
        imgs = img.unsqueeze(0).to(device)
        qs   = q_t.unsqueeze(0).to(device)
        pred = batch_greedy_decode(model, model_type, imgs, qs, vocab_a, max_len=40)
        ref  = ann.get('multiple_choice_answer','') + ' because ' + (ann.get('explanation',[''])[0])[:80]
        print(f'  Q : {ann["question"]}')
        print(f'  GT: {ref.strip()}')
        print(f'  PR: {pred[0].strip()}')
        print('  ' + '-' * 60)

# show_predictions('E', 'checkpoints/model_e_best.pth')
print('show_predictions() ready — provide a checkpoint to call it.')


## 6.4 Model Comparison Table


In [ ]:
import subprocess, os, re

models_to_compare = [m for m in ['A','B','C','D','E','F']
    if os.path.exists(f'checkpoints/model_{m.lower()}_best.pth')]

if models_to_compare:
    result = subprocess.run([
        'python', 'src/compare.py',
        '--models', ','.join(models_to_compare),
        '--epoch', '20',
    ], capture_output=True, text=True)
    print(result.stdout)
else:
    print('No checkpoints found yet. Train models first.')
    print('Expected: checkpoints/model_{a,b,c,d,e,f}_best.pth')


---
# 7. Advanced Techniques


## 7.1 Sequence Focal Loss (Tier D3)

Drop-in replacement for CrossEntropyLoss. Enable with `--focal --focal_gamma 2.0`.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from training.losses import SequenceFocalLoss
import torch.nn as nn

# Compare Focal vs CE on a sequence with rare and common tokens
torch.manual_seed(42)
B, T, V = 4, 20, 3000
logits  = torch.randn(B*T, V)
targets = torch.randint(0, V, (B*T,))
targets[::5] = 0   # 20% padding

ce_loss    = nn.CrossEntropyLoss(ignore_index=0)(logits, targets)
focal_loss = SequenceFocalLoss(gamma=2.0)(logits, targets)

print(f'CrossEntropyLoss  : {ce_loss.item():.4f}')
print(f'SequenceFocalLoss : {focal_loss.item():.4f}  (γ=2.0)')
print()
print('Focal loss is lower for well-predicted tokens (p_t high)')
print('and comparable for hard tokens (p_t low). Both converge to')
print('the same solution but Focal focuses gradient on hard tokens.')


## 7.2 Question-Type Curriculum (Tier D4)

Enable with `--curriculum`. Trains on easy question types first:
binary (0-25%) → color/count (25-50%) → what/where (50-75%) → why/how (full).


In [ ]:
import sys; sys.path.insert(0, 'src')
from training.curriculum import classify_question_type, CurriculumSampler
import matplotlib.pyplot as plt

# Show the question-type classification
examples = [
    'Is the dog running?',
    'Are there any trees?',
    'What color is the car?',
    'How many people are there?',
    'What is on the table?',
    'Where is the cat sitting?',
    'Which direction is the bus going?',
    'Why is the man smiling?',
    'How was the food prepared?',
]
tier_names = {0:'Binary', 1:'Color', 2:'Count', 3:'What', 4:'Where/Which', 5:'Why/How'}
print(f"{'Question':<45} Tier  Type")
print('-'*65)
for q in examples:
    t = classify_question_type(q)
    print(f'{q:<45}  {t}   {tier_names[t]}')


In [ ]:
# Simulate curriculum progress over 10 epochs
mock_anns = [{'question': q} for q in examples * 100]

from training.curriculum import compute_question_type_scores
scores  = compute_question_type_scores(mock_anns)
sampler = CurriculumSampler(scores, epoch=0, total_epochs=10)

print('Epoch  Stage   Pool size  Pct')
for ep in range(10):
    sampler.set_epoch(ep)
    pct = len(sampler) / len(scores) * 100
    stage = 'Binary' if ep < 2 else ('Color/Count' if ep < 5 else ('What/Where' if ep < 7 else 'Full'))
    print(f'  {ep+1:2d}   {stage:<12}  {len(sampler):>5}     {pct:.0f}%')


## 7.3 Mixed-Ratio Pretraining (Tier D2)

Enable with `--mix_vqa --mix_vqa_fraction 0.7` in Phase 1 only.

**Why**: VQA v2.0 answers are 1-3 tokens. Training on VQA v2.0 alone induces
length bias — the LSTM learns to emit `<end>` after ~2 tokens, contradicting
VQA-E's 5-20 token explanation objective.

**How**: WeightedRandomSampler gives VQA-E 3× oversampling relative to its
natural frequency in the combined dataset.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from torch.utils.data import Dataset, WeightedRandomSampler, ConcatDataset
from dataset import build_mixed_sampler

# Simulate the mixing behavior
class FakeDataset(Dataset):
    def __init__(self, n, label): self.n = n; self.label = label
    def __len__(self): return self.n
    def __getitem__(self, i): return i

vqa_v2  = FakeDataset(660_000, 'VQA v2.0')   # ~660K samples
vqa_e   = FakeDataset(210_000, 'VQA-E')       # ~210K samples

concat, sampler = build_mixed_sampler(vqa_v2, vqa_e, vqa_fraction=0.7)
print(f'ConcatDataset size  : {len(concat):,}')
print(f'Sampler num_samples : {sampler.num_samples:,}')
print(f'Weight VQA v2.0/sample : {0.7/660_000:.2e}  ({0.7:.0%} of batches)')
print(f'Weight VQA-E/sample    : {0.3/210_000:.2e}  ({0.3:.0%} of batches)')
print(f'VQA-E oversampling     : {(0.3/210_000) / (0.7/660_000):.1f}×')


## 7.4 ConceptNet GNN (Tier 9 — Optional)

Enriches word embeddings with commonsense knowledge from ConceptNet via GCN.
Falls back to a 2-layer MLP if `torch_geometric` is not installed.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from models.concept_gnn import ConceptGNN

# Demo: enrich word embeddings
gnn = ConceptGNN(vocab_size=8000, embed_dim=512, gcn_dim=256)

word_ids = torch.randint(4, 8000, (2, 15))   # (batch=2, seq_len=15)
enriched = gnn(word_ids)                      # (2, 15, 256)
print(f'Input  word_ids : {word_ids.shape}')
print(f'Output enriched : {enriched.shape}')
print()
print('Integration: pass to QuestionEncoder as enriched_emb argument')
print('after building the co-occurrence graph from training annotations.')


## 7.5 CSS Counterfactual Augmentation (Tier 6)

Enable with `--css --css_lambda 0.5`. Creates two counterfactual versions
of each batch to build visual and linguistic grounding:

- **Visual CF**: zero the top-k attended image regions → model must explain
  without the most attended object → forces multi-region reasoning
- **Linguistic CF**: replace content words in question with `<unk>` →
  model must not rely on question shortcuts

Loss = CE + λ_css · max(0, margin - ||f_real - f_cf||₂)


## 7.6 SCST Reinforcement Learning (Tier 8)

Enable with `--scst --scst_lambda 0.5` in Phase 4.

**Key insight**: CE loss rewards log-probability of the GT token at every
position, but BLEU-4 (the actual metric) depends on n-gram precision across
the full sequence. SCST directly optimizes the sequence-level metric:

```
r_sample = BLEU-4(model_sample, reference)   ← roll out stochastic decode
r_greedy = BLEU-4(greedy_output, reference)  ← baseline (no gradient)
REINFORCE loss = -(r_sample - r_greedy) · Σ_t log p(y_t^sample)
```

The baseline `r_greedy` reduces variance without biasing the gradient.


---
# 8. Full Model Benchmark


In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

# Load all available history files and display summary
summary = {}
for m in ['A','B','C','D','E','F']:
    path = f'checkpoints/history_model_{m.lower()}.json'
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        summary[m] = {
            'epochs'     : len(h['train_loss']),
            'best_val'   : min(h['val_loss']),
            'final_train': h['train_loss'][-1],
            'final_val'  : h['val_loss'][-1],
        }

if summary:
    print(f"{'Model':<8} {'Epochs':>6}  {'Best Val':>9}  {'Final Train':>12}  {'Final Val':>10}")
    print('-'*55)
    for m, s in sorted(summary.items()):
        print(f'  {m:<6} {s["epochs"]:>6}  {s["best_val"]:>9.4f}  {s["final_train"]:>12.4f}  {s["final_val"]:>10.4f}')
    best_m = min(summary, key=lambda m: summary[m]['best_val'])
    print(f'\nBest model: {best_m} (val loss = {summary[best_m]["best_val"]:.4f})')
else:
    print('No training history found yet.')
    print('Expected locations: checkpoints/history_model_{a..f}.json')


In [ ]:
# Architecture ablation reference table (fill in after evaluation)

ablation = {
    'Model A (SimpleCNN, no attn)':        {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Baseline'},
    'Model B (ResNet101, no attn)':        {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Pretrained encoder'},
    'Model C (SimpleCNN + MHCA)':          {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Spatial attention'},
    'Model D (ResNet101 + MHCA)':          {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Pretrained + attn'},
    'Model E (ConvNeXt + MUTAN + MHCA)':   {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Full best model'},
    'Model F (BUTD + MUTAN + MHCA)':       {'BLEU-4': '—', 'METEOR': '—', 'Notes': 'Object-centric'},
}

print(f"{'Architecture':<42} {'BLEU-4':>8}  {'METEOR':>8}  Notes")
print('-'*80)
for arch, m in ablation.items():
    print(f'{arch:<42} {m["BLEU-4"]:>8}  {m["METEOR"]:>8}  {m["Notes"]}')

print()
print('Fill in BLEU-4 and METEOR after running evaluate.py on each checkpoint.')


---
# 9. Inference API

Run the trained model on arbitrary images and questions.


In [ ]:
import sys; sys.path.insert(0, 'src')
import torch
from PIL import Image
from torchvision import transforms
from vocab import Vocabulary

def load_inference_model(model_type, checkpoint_path):
    """Load a trained model ready for inference."""
    from inference import load_model_from_checkpoint
    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
    vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')
    model   = load_model_from_checkpoint(
        model_type, checkpoint_path, len(vocab_q), len(vocab_a))
    model   = model.to(device).eval()
    return model, vocab_q, vocab_a, device

def answer_question(model, model_type, image_path, question_text,
                    vocab_q, vocab_a, device, beam_width=3):
    """Answer a question about an image."""
    from inference import batch_beam_search_decode_with_attention
    from inference import batch_greedy_decode
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    img = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    q_ids = torch.tensor(vocab_q.numericalize(question_text)).unsqueeze(0).to(device)

    if beam_width <= 1:
        answer = batch_greedy_decode(model, model_type, img, q_ids, vocab_a)
    else:
        answer = batch_beam_search_decode_with_attention(
            model, model_type, img, q_ids, vocab_a, beam_width=beam_width)
    return answer[0] if isinstance(answer, list) else answer

print('Inference API ready.')
print('Usage:')
print('  model, vocab_q, vocab_a, device = load_inference_model("E", "checkpoints/model_e_best.pth")')
print('  answer = answer_question(model, "E", "path/to/image.jpg", "What is in the image?", ...)')


---
# 10. Quick Reference — All train.py Flags

```bash
python src/train.py \
  # Core
  --model      {A,B,C,D,E,F}   # Model architecture
  --epochs     N                # Training epochs
  --lr         FLOAT            # Base learning rate
  --batch_size N                # Batch size
  --accum_steps N               # Gradient accumulation steps
  --resume     PATH             # Resume from checkpoint

  # Regularization
  --dropout       FLOAT         # Dropout rate (default 0.5)
  --weight_decay  FLOAT         # L2 regularization (default 1e-5)
  --grad_clip     FLOAT         # Gradient clipping norm (default 5.0)
  --label_smoothing FLOAT       # Label smoothing (non-focal, default 0.1)

  # Architecture flags
  --layer_norm                  # Tier 1A: LayerNorm inside LSTM
  --dropconnect                 # Tier 1B: DropConnect (AWD-LSTM)
  --coverage                    # Coverage mechanism (C/D/E/F)
  --coverage_lambda FLOAT       # λ for coverage loss (default 1.0)
  --use_mutan                   # Tier 4: MUTAN fusion (E/F)
  --pgn                         # Tier 5: Pointer-Generator
  --q_highway                   # Tier 7B: Highway BiLSTM
  --char_cnn                    # Tier 7C: Char-CNN embeddings

  # Training regime
  --augment                     # Image augmentation (RandAugment)
  --scheduled_sampling          # Phase 3: reduce exposure bias
  --ss_k       FLOAT            # SS decay speed (default 5.0)
  --scst                        # Phase 4: SCST RL
  --scst_lambda FLOAT           # λ for RL loss (default 0.5)
  --finetune_cnn                # Phase 2: unfreeze CNN backbone
  --cnn_lr_factor FLOAT         # Backbone LR = lr × factor (default 0.1)

  # Data tiers
  --focal                       # D3: SequenceFocalLoss
  --focal_gamma  FLOAT          # γ parameter (default 2.0)
  --curriculum                  # D4: question-type curriculum
  --mix_vqa                     # D2: mixed VQA v2.0 + VQA-E
  --mix_vqa_fraction FLOAT      # VQA v2.0 fraction (default 0.7)

  # CSS augmentation (Tier 6)
  --css                         # CSS counterfactual
  --css_lambda  FLOAT           # λ for CSS loss (default 0.5)
  --css_margin  FLOAT           # Hinge margin (default 1.0)

  # Model F
  --butd_feat_dir PATH          # Directory of pre-extracted .pt features

  # Misc
  --warmup_epochs N             # LR warmup (default 3, 0 for resume)
  --glove                       # GloVe 300d embeddings
  --early_stopping N            # Patience (default 0 = disabled)
  --no_compile                  # Disable torch.compile
  --num_workers N               # DataLoader workers
  --phase N                     # Phase number (1/2/3/4) for W&B

  # W&B
  --wandb                       # Enable Weights & Biases
  --wandb_project NAME          # W&B project (default: vqa-e)
  --wandb_run_name  NAME        # W&B run name
  --wandb_tags      TAGS        # Comma-separated tags
```


---

<p align='center'>
  <b>VQA-E Master Guide</b> — All-in-One Notebook<br>
  Models A–F · MHCA · MUTAN · PGN · CSS · SCST · Focal Loss · Curriculum
</p>
